In [1]:
%load_ext autoreload
%autoreload 2

In [14]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder

import torch
import torch_geometric as pyg

from bipartite_gnn.train_test import GNNTrainer

In [7]:
null_vals = ["NA"]
mrna = pl.read_csv("BRCA_PROCESSED_DATA/mrna.tsv", separator="\t", null_values=null_vals)
cna = pl.read_csv("BRCA_PROCESSED_DATA/cnvth.tsv", separator="\t", null_values=null_vals)
mirna = pl.read_csv("BRCA_PROCESSED_DATA/mirna.tsv", separator="\t", null_values=null_vals)

labels = pl.read_csv("BRCA_PROCESSED_DATA/labels.tsv", separator="\t")
le = LabelEncoder()
le.fit(labels["PAM50_mRNA_nature2012"].to_list())
y = le.transform(labels["PAM50_mRNA_nature2012"].to_list())
# labels, y

In [43]:
# mrna, cna, mirna
mrna_gene_names = mrna[:, 0].to_list()
cna_gene_names = cna[:, 0].to_list()
mirna_gene_names = mirna[:, 0].to_list()

In [44]:
mrna_X = torch.tensor(mrna[:, 1:].to_numpy().T)
cna_X = torch.tensor(cna[:, 1:].to_numpy().T)
mirna_X = torch.tensor(mirna[:, 1:].to_numpy().T)

In [45]:
mrna_X.shape

torch.Size([483, 18975])

In [46]:
# create hetero-data object
data = pyg.data.HeteroData()

proj_dim = 128

# sample node features
data["mrna"].x = mrna_X
data["cna"].x = cna_X
data["mirna"].x = mirna_X
data["mrna"].y = data["cna"].y = data["mirna"].y = torch.tensor(y)

# feature node projection
data["mrna_feature"].x = torch.ones(mrna_X.shape[1], proj_dim)
data["cna_feature"].x = torch.ones(cna_X.shape[1], proj_dim)
data["mirna_feature"].x = torch.ones(mirna_X.shape[1], proj_dim)

dh = data.to_homogeneous()

In [47]:
data

HeteroData(
  mrna={
    x=[483, 18975],
    y=[483],
  },
  cna={
    x=[483, 24776],
    y=[483],
  },
  mirna={
    x=[483, 231],
    y=[483],
  },
  mrna_feature={ x=[18975, 128] },
  cna_feature={ x=[24776, 128] },
  mirna_feature={ x=[231, 128] }
)

In [48]:
mrna_X.shape, cna_X.shape, mirna_X.shape

(torch.Size([483, 18975]), torch.Size([483, 24776]), torch.Size([483, 231]))

In [49]:
dh

Data(x=[45431, 24776], y=[45431], node_type=[45431])

In [50]:
# get counts of unique values in the dh.node_type tensor
node_type_counts = torch.unique(dh.node_type, return_counts=True)
node_type_counts

(tensor([0, 1, 2, 3, 4, 5]),
 tensor([  483,   483,   483, 18975, 24776,   231]))

In [19]:
data

HeteroData(
  mrna={
    x=[18975, 483],
    y=[483],
  },
  cna={
    x=[24776, 483],
    y=[483],
  },
  mirna={
    x=[231, 483],
    y=[483],
  }
)

In [ ]:
# train the model